In [1]:
# scraping_lora_fixed.py
import torch
from newspaper import Article
from datasets import Dataset
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM,DataCollatorForSeq2Seq,Seq2SeqTrainer,Seq2SeqTrainingArguments
from peft import LoraConfig, get_peft_model, PeftModel

# ---------- Config ----------
URL = "https://www.cnbc.com/2025/09/11/solomon-dimon-pick-ceos-economy.html"
BASE_MODEL = "sshleifer/distilbart-cnn-12-6"   # CPU-friendly small seq2seq
OUTPUT_DIR = "./lora-finance"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ENCODER_MAX_LEN = 1024    # BART-family encoder max tokens
SUMMARY_MAX_LEN = 128     # target summary length
# ----------------------------

# 1) Scrape article
article = Article(URL)
article.download()
article.parse()
article.nlp()

# 2) Build dataset 
dataset = Dataset.from_dict({"content": [article.text], "summary": [article.summary]})

# 3) Tokenizer & preprocess (note truncation to encoder max length)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def preprocess(batch):
    model_inputs = tokenizer(
        batch["article"],
        truncation=True,
        padding="max_length",
        max_length=ENCODER_MAX_LEN,
    )
    # tokenize targets (summaries)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["summary"],
            truncation=True,
            padding="max_length",
            max_length=SUMMARY_MAX_LEN,
        )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = dataset.map(preprocess, batched=True, remove_columns=["article", "summary"])

# 4) Load base model (object) and wrap with LoRA
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
base_model.to(DEVICE)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM",
)

model = get_peft_model(base_model, lora_config)
model.to(DEVICE)

# 5) Data collator and training args
data_collator = DataCollatorForSeq2Seq(tokenizer, model=base_model)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    num_train_epochs=1,
    learning_rate=5e-4,
    logging_dir="./logs",
    save_strategy="no",   # demo: do not create many checkpoints
    eval_strategy="no",
    fp16=False if DEVICE == "cpu" else True,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# 6) Train (LoRA adapters will be optimized)
print("Starting training (demo 1 epoch)...")
trainer.train()
print("Training finished.")

# 7) Save LoRA adapter properly (writes adapter_config.json + weights)
# Use model.save_pretrained so peft can reload it later
model.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter saved to {OUTPUT_DIR}")

# 8) Reload base model + LoRA adapter for inference (clean load)
# (This avoids warnings about duplicate adapters)
base_for_infer = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
model_for_infer = PeftModel.from_pretrained(base_for_infer, OUTPUT_DIR)
model_for_infer.to(DEVICE)

# 9) Inference: manually tokenize with truncation to avoid >1024 tokens
#    (pipeline would try to pass full text without explicit truncation)
encoded = tokenizer(
    article.text,
    truncation=True,
    max_length=ENCODER_MAX_LEN,
    return_tensors="pt",
)

encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

# Generate summary (use generate directly)
with torch.no_grad():
    generated_ids = model_for_infer.generate(
        input_ids=encoded["input_ids"],
        attention_mask=encoded["attention_mask"],
        max_length=120,
        min_length=30,
        num_beams=4,
        early_stopping=True,
    )

summary_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print("\n--- FINAL OUTPUTS ---")
print("Title:", article.title)
print("Article:\n",article.text)
print("\nOriginal newspaper3k summary:\n", article.summary)
print("\nGenerated LoRA summary:\n", summary_text)

ImportError: lxml.html.clean module is now a separate project lxml_html_clean.
Install lxml[html_clean] or lxml_html_clean directly.